In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def running_in_colab():
    return importlib.util.find_spec("google.colab") is not None


def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name])
    return __import__(import_name)


IS_COLAB = running_in_colab()
duckdb = ensure_package("duckdb")
pd = ensure_package("pandas")
matplotlib = ensure_package("matplotlib")
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")
con = duckdb.connect()


def query_df(query):
    return con.sql(query).df()


def null_rate_table(table_ref, dataset_name, columns):
    statements = [
        f"SELECT '{dataset_name}' AS dataset, '{column}' AS column_name, count(*) FILTER (WHERE {column} IS NULL) AS null_count, count(*) AS row_count FROM {table_ref}"
        for column in columns
    ]
    df = query_df("\nUNION ALL\n".join(statements))
    df["null_rate_pct"] = (100.0 * df["null_count"] / df["row_count"]).round(4)
    return df.sort_values(["dataset", "column_name"]).reset_index(drop=True)


def distinct_count_table(table_ref, features):
    statements = [
        f"SELECT '{label}' AS feature, count(DISTINCT {expr}) AS distinct_count FROM {table_ref}"
        for label, expr in features
    ]
    return query_df("\nUNION ALL\n".join(statements))


print(f"Running in Colab: {IS_COLAB}")
print(f"Python executable: {sys.executable}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")


Running in Colab: False
Python executable: /Users/harrish/Desktop/practicum/.venv/bin/python
DuckDB version: 1.5.2
Pandas version: 3.0.3
Matplotlib version: 3.10.9


In [2]:
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    dataset_root = Path("/content/drive/MyDrive/Practicum/NGIDS/NGIDS-DS-v1/parquet")
else:
    dataset_root = Path("../dataset")
    if not dataset_root.exists():
        dataset_root = Path("dataset")
host_logs = dataset_root / "host_logs.parquet"
ground_truth = dataset_root / "ground_truth.parquet"
syscall_lookup = dataset_root / "syscall-lookup-linux-v3_13.csv"
print(f"dataset_root: {dataset_root}")
print(f"host_logs parquet: {host_logs}")
print(f"ground_truth parquet: {ground_truth}")
print(f"syscall lookup: {syscall_lookup}")


dataset_root: ../dataset
host_logs parquet: ../dataset/host_logs.parquet
ground_truth parquet: ../dataset/ground_truth.parquet
syscall lookup: ../dataset/syscall-lookup-linux-v3_13.csv


## Phase 1: Dataset Orientation

This section verifies row counts, schemas, sample rows, time ranges, and a few label-like columns.


In [3]:
query_df(f"""
SELECT 'host_logs' AS dataset, count(*) AS row_count FROM '{host_logs}'
UNION ALL
SELECT 'ground_truth' AS dataset, count(*) AS row_count FROM '{ground_truth}'
""")


,dataset,row_count
0,host_logs,90054239
1,ground_truth,313926


**Inference.** `host_logs` is the main analytical table with about 90.1 million rows, while `ground_truth` is much smaller at about 313.9 thousand rows. That size gap is why the modeling pipeline should start from `host_logs` and treat `ground_truth` as supporting metadata.


In [4]:
query_df(f"DESCRIBE SELECT * FROM '{host_logs}'")


,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,TIME,YES,None,None,None
2,pro_id,BIGINT,YES,None,None,None
3,path,VARCHAR,YES,None,None,None
4,sys_call,BIGINT,YES,None,None,None
5,event_id,BIGINT,YES,None,None,None
6,attack_cat,VARCHAR,YES,None,None,None
7,attack_subcat,VARCHAR,YES,None,None,None
8,label,BIGINT,YES,None,None,None


**Inference.** `host_logs` already exposes `attack_cat`, `attack_subcat`, and `label`. Those columns are useful for audit, but they are direct leakage channels for supervised modeling.


In [5]:
query_df(f"SELECT * FROM '{host_logs}' LIMIT 10")


,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label
0,2016-03-11,02:45:01,1830,/sbin/upstart-dbus-bridge,142,45354,normal,normal,0
1,2016-03-11,02:45:06,1804,/bin/dbus-daemon,256,45352,normal,normal,0
2,2016-03-11,02:45:06,2133,/usr/lib/i386-linux-gnu/gconf/gconfd-2,168,45372,normal,normal,0
3,2016-03-11,02:45:35,4528,/usr/bin/python3.4,3,39459,normal,normal,0
4,2016-03-11,02:45:44,1847,/usr/bin/ibus-daemon,102,37263,normal,normal,0
5,2016-03-11,02:45:44,1907,/usr/lib/ibus/ibus-ui-gtk3,168,37896,normal,normal,0
6,2016-03-11,02:45:44,1925,/usr/lib/ibus/ibus-engine-simple,168,37542,normal,normal,0
7,2016-03-11,02:45:44,4461,/usr/sbin/apache2,142,37647,normal,normal,0
8,2016-03-11,02:45:45,1081,/usr/bin/Xorg,102,37480,normal,normal,0
9,2016-03-11,02:45:11,3989,/sbin/auditd,256,45374,normal,normal,0


**Inference.** The sample rows confirm that `host_logs` is an event-level process activity table anchored on executable path, process id, and system call behavior.


In [6]:
query_df(f"""
SELECT
    min(date) AS min_date,
    max(date) AS max_date,
    min(time) AS min_time,
    max(time) AS max_time,
    count(*) AS row_count,
    count(DISTINCT row(date, time, pro_id, path, sys_call, event_id, attack_cat, attack_subcat, label)) AS distinct_rows,
    count(*) - count(DISTINCT row(date, time, pro_id, path, sys_call, event_id, attack_cat, attack_subcat, label)) AS duplicate_rows
FROM '{host_logs}'
""")


,min_date,max_date,min_time,max_time,row_count,distinct_rows,duplicate_rows
0,2016-03-11,2016-03-16,00:00:00,23:59:59,90054239,89709995,344244


**Inference.** `host_logs` spans `2016-03-11` to `2016-03-16` and contains duplicate rows. Time coverage and repeat activity both need to be handled explicitly later in the pipeline.


In [7]:
query_df(f"""
SELECT attack_cat, attack_subcat, label, count(*) AS n
FROM '{host_logs}'
GROUP BY 1, 2, 3
ORDER BY n DESC
LIMIT 20
""")


,attack_cat,attack_subcat,label,n
0,normal,normal,0,88791812
1,Exploits,Office Document Batch,1,276578
2,Exploits,Browser,1,152319
3,Exploits,Clientside,1,102893
4,Generic,IXIA Batch,1,79624
5,Exploits,Clientside Microsoft Office Batch,1,71920
6,Exploits,Clientside Microsoft Paint,1,71869
7,Backdoors,All Batch,1,70712
8,Exploits,Browser FTP Batch,1,48994
9,Shellcode,Linux Batch,1,44245


**Inference.** The binary label and richer attack metadata already move together inside `host_logs`. That is convenient for audit, but unsafe to feed directly into a model.


In [8]:
query_df(f"""
SELECT
    count(DISTINCT pro_id) AS distinct_pro_id,
    count(DISTINCT path) AS distinct_path,
    count(DISTINCT sys_call) AS distinct_sys_call,
    count(DISTINCT event_id) AS distinct_event_id,
    count(DISTINCT attack_cat) AS distinct_attack_cat,
    count(DISTINCT attack_subcat) AS distinct_attack_subcat,
    count(DISTINCT label) AS distinct_label
FROM '{host_logs}'
""")


,distinct_pro_id,distinct_path,distinct_sys_call,distinct_event_id,distinct_attack_cat,distinct_attack_subcat,distinct_label
0,5576,100,122,89709941,8,53,2


**Inference.** `event_id` behaves like an identifier, `path` and `sys_call` stay at workable cardinalities, and `pro_id` is high-cardinality but still suitable for aggregate features.


In [9]:
query_df(f"DESCRIBE SELECT * FROM '{ground_truth}'")


,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,VARCHAR,YES,None,None,None
2,attack_cat,VARCHAR,YES,None,None,None
3,attack_subcat,VARCHAR,YES,None,None,None
4,attack_name,VARCHAR,YES,None,None,None
5,attack_refrence,VARCHAR,YES,None,None,None
6,ips,VARCHAR,YES,None,None,None


**Inference.** `ground_truth` looks like a smaller metadata table with time, attack family, strike description, and IP tuple fields rather than a direct host-event mirror.


In [10]:
query_df(f"SELECT * FROM '{ground_truth}' LIMIT 10")


,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,9:21:36,Backdoors,All Batch,Cisco Network Registrar Default Credentials Ba...,CVE 2011-2024 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.0:0->10.40.85.32:0 175.45.176.0:...
1,2016-03-14,12:14:24,Backdoors,All Batch,BlackEnergy Botnet Command and Control Communi...,http://atlas-public.ec2.arbor.net/docs/BlackEn...,175.45.176.1:3495->10.40.85.32:58782
2,2016-03-15,4:19:12,Backdoors,All Batch,Backdoor: Cisco Prime LAN Management (https://...,CVE 2012-6392 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:13177->10.40.85.32:514
3,2016-03-14,9:36:00,Backdoors,All Batch,phpmyadmin 3.5.2.2 Backdoor Access and Code Ex...,CVE 2012-5159 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.2:0->10.40.85.32:0 175.45.176.2:...
4,2016-03-15,11:45:36,Backdoors,All Batch,Android AndroidKungFu Malware Command and Cont...,http://about-threats.trendmicro.com/malware.as...,175.45.176.3:61508->10.40.85.32:7500
5,2016-03-11,9:21:36,Backdoors,All Batch,Cisco Network Registrar Default Credentials Ba...,CVE 2011-2024 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.0:0->10.40.85.32:0 175.45.176.0:...
6,2016-03-14,12:57:36,Backdoors,All Batch,BlackEnergy Botnet Command and Control Communi...,http://atlas-public.ec2.arbor.net/docs/BlackEn...,175.45.176.2:30436->10.40.85.32:30566
7,2016-03-14,12:14:24,Backdoors,All Batch,Backdoor: Cisco Prime LAN Management (https://...,CVE 2012-6392 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:59379->10.40.85.32:514
8,2016-03-16,2:09:36,Backdoors,All Batch,phpmyadmin 3.5.2.2 Backdoor Access and Code Ex...,CVE 2012-5159 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.0:0->10.40.85.32:0 175.45.176.0:...
9,2016-03-15,6:28:48,Backdoors,All Batch,Android AndroidKungFu Malware Command and Cont...,http://about-threats.trendmicro.com/malware.as...,175.45.176.1:49989->10.40.85.32:7500


**Inference.** The sample rows reinforce that `ground_truth` is attack annotation data. It does not read like a clean one-to-one companion table for host-log events.


In [11]:
query_df(f"""
SELECT
    min(date) AS min_date,
    max(date) AS max_date,
    count(*) AS row_count,
    count(DISTINCT row(date, time, attack_cat, attack_subcat, attack_name, attack_refrence, ips)) AS distinct_rows,
    count(*) - count(DISTINCT row(date, time, attack_cat, attack_subcat, attack_name, attack_refrence, ips)) AS duplicate_rows,
    count(*) FILTER (WHERE time = 'Time') AS header_like_rows
FROM '{ground_truth}'
""")


,min_date,max_date,row_count,distinct_rows,duplicate_rows,header_like_rows
0,2016-03-11,2016-03-16,313926,311621,2305,26


**Inference.** `ground_truth` covers the same calendar window as `host_logs`, but it also contains duplicate rows before any cleaning is applied.


In [12]:
query_df(f"""
SELECT attack_cat, count(*) AS n
FROM '{ground_truth}'
GROUP BY 1
ORDER BY n DESC
LIMIT 20
""")


,attack_cat,n
0,Exploits,158316
1,Exploits,73301
2,Malware,35903
3,Denial of Service,18702
4,Generic,11300
5,Denial of Service,6100
6,Shellcode,5302
7,Reconnaissance,1900
8,Worms,1301
9,Backdoors,1200


**Inference.** The raw category mix is already noisy enough to justify a dedicated quality pass before `ground_truth` is used downstream.


In [13]:
query_df(f"SELECT * FROM '{ground_truth}' WHERE time = 'Time' LIMIT 10")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,Time,Malware,Mobile Batch,Strike Name,Strike Reference,Strike Tuples
1,2016-03-11,Time,Denial of Service,Browser Batch,Strike Name,Strike Reference,Strike Tuples
2,2016-03-11,Time,Denial of Service,HTTP,Strike Name,Strike Reference,Strike Tuples
3,2016-03-11,Time,Malware,package,Strike Name,Strike Reference,Strike Tuples
4,2016-03-11,Time,Denial of Service,NetBIOS/SMB Batch,Strike Name,Strike Reference,Strike Tuples
5,2016-03-11,Time,Exploits,Browser,Strike Name,Strike Reference,Strike Tuples
6,2016-03-11,Time,Exploits,Browser,Strike Name,Strike Reference,Strike Tuples
7,2016-03-11,Time,Exploits,Browser,Strike Name,Strike Reference,Strike Tuples
8,2016-03-11,Time,Exploits,Browser,Strike Name,Strike Reference,Strike Tuples
9,2016-03-11,Time,Exploits,Clientside Microsoft Office Batch,Strike Name,Strike Reference,Strike Tuples


**Inference.** Header-like rows are embedded inside the dataset itself. This confirms source-data contamination rather than a parquet-only conversion issue.


## Phase 2: Data Quality Baseline

This section measures nulls, blanks, whitespace issues, duplicate behavior, builds a flagged quality view for `ground_truth.parquet`, and creates a filtered `ground_truth_clean` subset that excludes corrupted rows.


In [14]:
query_df(f"""
SELECT
    count(*) AS row_count,
    count(*) FILTER (WHERE date IS NULL) AS null_date,
    count(*) FILTER (WHERE time IS NULL) AS null_time,
    count(*) FILTER (WHERE pro_id IS NULL) AS null_pro_id,
    count(*) FILTER (WHERE path IS NULL) AS null_path,
    count(*) FILTER (WHERE trim(coalesce(path, '')) = '') AS blank_path,
    count(*) FILTER (WHERE sys_call IS NULL) AS null_sys_call,
    count(*) FILTER (WHERE event_id IS NULL) AS null_event_id,
    count(*) FILTER (WHERE attack_cat IS NULL) AS null_attack_cat,
    count(*) FILTER (WHERE attack_subcat IS NULL) AS null_attack_subcat,
    count(*) FILTER (WHERE label IS NULL) AS null_label,
    count(*) FILTER (WHERE pro_id < 0) AS negative_pro_id,
    count(*) FILTER (WHERE sys_call < 0) AS negative_sys_call,
    count(*) FILTER (WHERE sys_call = 0) AS zero_sys_call,
    count(*) FILTER (WHERE event_id < 0) AS negative_event_id,
    count(*) FILTER (WHERE label NOT IN (0, 1)) AS invalid_label,
    count(*) FILTER (WHERE attack_subcat <> trim(attack_subcat)) AS attack_subcat_with_outer_spaces
FROM '{host_logs}'
""")


,row_count,null_date,null_time,null_pro_id,null_path,blank_path,null_sys_call,null_event_id,null_attack_cat,null_attack_subcat,null_label,negative_pro_id,negative_sys_call,zero_sys_call,negative_event_id,invalid_label,attack_subcat_with_outer_spaces
0,90054239,0,0,0,0,0,0,0,0,0,0,0,0,573,0,0,495005


**Inference.** `host_logs` is structurally clean overall. Its main quality issue is not nulls, but duplicate rows and a small amount of categorical whitespace.


In [15]:
query_df(f"""
SELECT
    count(DISTINCT attack_cat) AS raw_attack_cat_count,
    count(DISTINCT trim(attack_cat)) AS trimmed_attack_cat_count,
    count(DISTINCT attack_subcat) AS raw_attack_subcat_count,
    count(DISTINCT trim(attack_subcat)) AS trimmed_attack_subcat_count
FROM '{host_logs}'
""")


,raw_attack_cat_count,trimmed_attack_cat_count,raw_attack_subcat_count,trimmed_attack_subcat_count
0,8,8,53,45


**Inference.** Whitespace materially changes `attack_subcat` cardinality, so trimming is necessary before any category-level summaries are trusted.


In [16]:
query_df(f"""
SELECT attack_subcat, trim(attack_subcat) AS trimmed_attack_subcat, count(*) AS n
FROM '{host_logs}'
WHERE attack_subcat <> trim(attack_subcat)
GROUP BY 1, 2
ORDER BY n DESC
LIMIT 20
""")


,attack_subcat,trimmed_attack_subcat,n
0,Clientside,Clientside,102893
1,IXIA Batch,IXIA Batch,79624
2,Clientside Microsoft Paint,Clientside Microsoft Paint,71869
3,Browser FTP Batch,Browser FTP Batch,48994
4,Microsoft IIS Batch,Microsoft IIS Batch,33509
5,SMB Batch,SMB Batch,23225
6,Clientside Microsoft Office Batch,Clientside Microsoft Office Batch,21470
7,Browser,Browser,17271
8,NetBIOS/SMB Batch,NetBIOS/SMB Batch,16823
9,All Batch,All Batch,14125


**Inference.** These examples show a normalization problem rather than a semantic one. Trimming outer whitespace is a safe cleanup step here.


In [17]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW ground_truth_flagged AS
SELECT
    date,
    time,
    attack_cat,
    attack_subcat,
    attack_name,
    attack_refrence,
    ips,
    CASE
        WHEN time = 'Time' THEN 'header_row'
        WHEN instr(trim(attack_cat), '->') > 0 THEN 'shift_pattern_a'
        WHEN instr(trim(time), '->') > 0 THEN 'shift_pattern_b'
        ELSE 'normal'
    END AS row_pattern
FROM '{ground_truth}'
""")



**Inference.** This helper step separates raw `ground_truth` corruption patterns from the filtered `ground_truth_clean` subset used later in the notebook.


In [18]:
query_df(f"""
SELECT
    count(*) AS row_count,
    count(*) FILTER (WHERE time = 'Time') AS header_like_rows,
    count(*) FILTER (WHERE attack_cat <> trim(attack_cat)) AS attack_cat_with_outer_spaces,
    count(*) FILTER (WHERE attack_subcat <> trim(attack_subcat)) AS attack_subcat_with_outer_spaces,
    count(DISTINCT attack_cat) AS raw_attack_cat_count,
    count(DISTINCT trim(attack_cat)) AS trimmed_attack_cat_count,
    count(DISTINCT attack_subcat) AS raw_attack_subcat_count,
    count(DISTINCT trim(attack_subcat)) AS trimmed_attack_subcat_count
FROM '{ground_truth}'
""")


,row_count,header_like_rows,attack_cat_with_outer_spaces,attack_subcat_with_outer_spaces,raw_attack_cat_count,trimmed_attack_cat_count,raw_attack_subcat_count,trimmed_attack_subcat_count
0,313926,26,177418,122806,357,355,130,118


**Inference.** `ground_truth` has no true null problem either. Its quality issues are header contamination, whitespace pollution, and shifted rows.


In [20]:
query_df("""
WITH flagged_summary AS (
    SELECT
        count(*) AS total_rows,
        count(*) FILTER (WHERE row_pattern = 'header_row') AS header_rows,
        count(*) FILTER (WHERE row_pattern = 'shift_pattern_a') AS shift_pattern_a_rows,
        count(*) FILTER (WHERE row_pattern = 'shift_pattern_b') AS shift_pattern_b_rows,
        count(*) FILTER (WHERE row_pattern <> 'normal') AS removed_rows
    FROM ground_truth_flagged
),
clean_summary AS (
    SELECT
        count(*) AS cleaned_rows,
        count(DISTINCT row(date, time, attack_cat, attack_subcat, attack_name, attack_refrence, ips)) AS cleaned_distinct_rows,
        count(*) - count(DISTINCT row(date, time, attack_cat, attack_subcat, attack_name, attack_refrence, ips)) AS cleaned_duplicate_rows,
        count(DISTINCT attack_cat) AS cleaned_attack_cat_count,
        count(DISTINCT attack_subcat) AS cleaned_attack_subcat_count
    FROM ground_truth_clean
)
SELECT *
FROM flagged_summary
CROSS JOIN clean_summary
""")


CatalogException: Catalog Error: Table with name ground_truth_clean does not exist!
Did you mean "ground_truth_flagged"?

**Inference.** After removing header and shifted rows, `ground_truth_clean` becomes much more trustworthy for alignment checks. The retained attack families remain intact after cleaning.


In [ ]:
query_df("""
SELECT attack_cat, count(*) AS n
FROM ground_truth_clean
GROUP BY 1
ORDER BY n DESC
LIMIT 20
""")


**Inference.** Cleaning removes corruption, not signal. The major attack families still dominate the filtered subset.


In [ ]:
query_df("""
SELECT
    date,
    row_pattern,
    time,
    attack_cat,
    attack_subcat,
    attack_name,
    attack_refrence,
    ips
FROM ground_truth_flagged
WHERE row_pattern <> 'normal'
LIMIT 20
""")


**Inference.** The malformed-row examples make the failure mode explicit. These records should be excluded, not repaired into labels.


In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW host_logs_anomalies AS
SELECT
    date,
    CAST(time AS VARCHAR) AS time,
    trim(attack_cat) AS attack_cat,
    trim(attack_subcat) AS attack_subcat
FROM '{host_logs}'
WHERE label = 1
""")


**Inference.** This helper view isolates anomalous host-log keys so the alignment cells below stay readable and consistent.


## Phase 3: Label And Ground Truth Mapping

This section checks whether the filtered `ground_truth_clean` subset can be aligned directly with anomalous `host_logs` rows, or whether it is better treated as supporting metadata.


In [ ]:
query_df("""
WITH gt_summary AS (
    SELECT
        count(*) AS gt_rows,
        count(DISTINCT row(date, time, attack_cat, attack_subcat)) AS gt_distinct_keys
    FROM ground_truth_clean
),
hl_summary AS (
    SELECT
        count(*) AS host_anomaly_rows,
        count(DISTINCT row(date, time, attack_cat, attack_subcat)) AS host_anomaly_distinct_keys
    FROM host_logs_anomalies
),
exact_match_keys AS (
    SELECT DISTINCT gt.date, gt.time, gt.attack_cat, gt.attack_subcat
    FROM ground_truth_clean gt
    JOIN host_logs_anomalies hl USING (date, time, attack_cat, attack_subcat)
),
timestamp_match_keys AS (
    SELECT DISTINCT gt.date, gt.time
    FROM ground_truth_clean gt
    JOIN host_logs_anomalies hl USING (date, time)
),
match_summary AS (
    SELECT
        (SELECT count(*) FROM ground_truth_clean gt JOIN exact_match_keys em USING (date, time, attack_cat, attack_subcat)) AS gt_rows_with_exact_host_match,
        (SELECT count(*) FROM host_logs_anomalies hl JOIN exact_match_keys em USING (date, time, attack_cat, attack_subcat)) AS host_anomaly_rows_with_exact_gt_match,
        (SELECT count(*) FROM ground_truth_clean gt JOIN timestamp_match_keys tm USING (date, time)) AS gt_rows_with_timestamp_match,
        (SELECT count(*) FROM host_logs_anomalies hl JOIN timestamp_match_keys tm USING (date, time)) AS host_anomaly_rows_with_timestamp_match
)
SELECT *
FROM gt_summary
CROSS JOIN hl_summary
CROSS JOIN match_summary
""")


**Inference.** Exact overlap exists, but it is limited. `ground_truth` only partially aligns with anomalous host-log events, so it is better treated as supporting metadata than as a required training join.


In [ ]:
query_df("""
WITH exact_match_keys AS (
    SELECT DISTINCT gt.date, gt.time, gt.attack_cat, gt.attack_subcat
    FROM ground_truth_clean gt
    JOIN host_logs_anomalies hl USING (date, time, attack_cat, attack_subcat)
)
SELECT
    gt.attack_cat,
    count(*) AS gt_rows,
    count(em.attack_cat) AS gt_rows_with_exact_match
FROM ground_truth_clean gt
LEFT JOIN exact_match_keys em USING (date, time, attack_cat, attack_subcat)
GROUP BY 1
ORDER BY gt_rows DESC
""")


**Inference.** The exact matches are concentrated in a small subset of attack families rather than spread evenly across all categories.


## Phase 4: Host Log Temporal And Entity Profiling

This section profiles when anomalies occur and how they concentrate across `path`, `sys_call`, and `pro_id`. The goal is to identify behavioral signals that are useful for modeling without depending on `ground_truth.parquet`.


In [ ]:
query_df(f"""
SELECT
    date,
    count(*) AS event_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct
FROM '{host_logs}'
GROUP BY 1
ORDER BY 1
""")


**Inference.** Anomaly rate changes by date and trends upward toward the end of the capture window. That is one reason the baseline split should stay chronological.


In [ ]:
query_df(f"""
SELECT
    extract('hour' FROM time) AS hour,
    count(*) AS event_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct
FROM '{host_logs}'
GROUP BY 1
ORDER BY 1
""")


**Inference.** Hourly behavior is highly structured. Anomalies are not uniformly distributed across the day.


In [ ]:
query_df(f"""
SELECT
    CASE
        WHEN extract('hour' FROM time) BETWEEN 1 AND 12 THEN '01-12'
        ELSE 'other_hours'
    END AS hour_block,
    count(*) AS event_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct
FROM '{host_logs}'
GROUP BY 1
ORDER BY 1
""")


**Inference.** The `01-12` hour block captures the full anomaly mass. This is a powerful shortcut the model could exploit if evaluation is careless.


In [ ]:
query_df(f"""
WITH path_stats AS (
    SELECT
        path,
        count(*) AS event_count,
        count(*) FILTER (WHERE label = 1) AS anomaly_count,
        100.0 * count(*) FILTER (WHERE label = 1) / count(*) AS anomaly_rate_pct
    FROM '{host_logs}'
    GROUP BY 1
), totals AS (
    SELECT count(*) FILTER (WHERE label = 1) AS total_anomalies
    FROM '{host_logs}'
)
SELECT
    count(*) AS total_paths,
    count(*) FILTER (WHERE anomaly_count = 0) AS zero_anomaly_paths,
    count(*) FILTER (WHERE anomaly_rate_pct = 100.0) AS all_anomaly_paths,
    count(*) FILTER (WHERE anomaly_rate_pct >= 95.0) AS near_all_anomaly_paths,
    round(100.0 * sum(anomaly_count) FILTER (WHERE path IN ('/usr/lib/firefox/firefox', '/usr/lib/libreoffice/program/soffice.bin')) / max(total_anomalies), 4) AS firefox_soffice_anomaly_share_pct
FROM path_stats, totals
""")


**Inference.** Path-level concentration is extreme: many paths never carry anomalies, while a few executables dominate them.


In [ ]:
query_df(f"""
WITH totals AS (
    SELECT count(*) FILTER (WHERE label = 1) AS total_anomalies
    FROM '{host_logs}'
)
SELECT
    path,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / max(total_anomalies), 4) AS anomaly_share_pct,
    count(*) AS event_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct
FROM '{host_logs}', totals
GROUP BY 1
ORDER BY anomaly_count DESC, event_count DESC
LIMIT 15
""")


**Inference.** A small number of executables contribute a disproportionate share of anomalous rows, especially `firefox` and `soffice.bin`.


In [ ]:
query_df(f"""
SELECT
    sys_call,
    count(*) AS event_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct,
    count(DISTINCT path) AS distinct_paths
FROM '{host_logs}'
GROUP BY 1
HAVING count(*) >= 100000
ORDER BY anomaly_rate_pct DESC, event_count DESC
LIMIT 20
""")


**Inference.** `sys_call` has signal, but its standalone anomaly rates are weaker than the path-driven effects.


In [ ]:
query_df(f"""
SELECT
    path,
    sys_call,
    count(*) AS event_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct
FROM '{host_logs}'
GROUP BY 1, 2
HAVING count(*) >= 10000
ORDER BY anomaly_rate_pct DESC, event_count DESC
LIMIT 20
""")


**Inference.** Order-agnostic interactions like `path x sys_call` are already far stronger than raw `sys_call`, which is why they belong in the baseline feature set.


In [ ]:
query_df(f"""
SELECT
    pro_id,
    count(*) AS event_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct,
    count(DISTINCT path) AS distinct_paths,
    count(DISTINCT sys_call) AS distinct_sys_calls
FROM '{host_logs}'
GROUP BY 1
HAVING count(*) >= 10000
ORDER BY anomaly_rate_pct DESC, event_count DESC
LIMIT 20
""")


**Inference.** `pro_id` carries signal, but it behaves like a local execution handle rather than a stable semantic identity. Aggregate features are safer than memorizing the raw id.


## Phase 5: Feature Readiness And Split Strategy

This section turns the host-log EDA into a concrete modeling direction. It recommends a chronological split, checks holdout coverage against training, and records the baseline feature shortlist.


In [ ]:
query_df(f"""
SELECT
    CASE
        WHEN date <= DATE '2016-03-14' THEN 'train'
        WHEN date = DATE '2016-03-15' THEN 'validation'
        ELSE 'test'
    END AS split_part,
    count(*) AS rows,
    count(*) FILTER (WHERE label = 1) AS anomalies,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct,
    count(DISTINCT path) AS distinct_paths,
    count(DISTINCT sys_call) AS distinct_sys_calls,
    count(DISTINCT pro_id) AS distinct_pro_ids,
    count(DISTINCT row(path, sys_call)) AS distinct_path_sys_calls
FROM '{host_logs}'
GROUP BY 1
ORDER BY CASE split_part WHEN 'train' THEN 1 WHEN 'validation' THEN 2 ELSE 3 END
""")


**Inference.** The chosen split keeps strong anomaly support in every partition while preserving chronological order.


In [ ]:
query_df(f"""
WITH train_paths AS (
    SELECT DISTINCT path FROM '{host_logs}' WHERE date <= DATE '2016-03-14'
),
train_sys_calls AS (
    SELECT DISTINCT sys_call FROM '{host_logs}' WHERE date <= DATE '2016-03-14'
),
train_path_sys_calls AS (
    SELECT DISTINCT path, sys_call FROM '{host_logs}' WHERE date <= DATE '2016-03-14'
),
train_pro_ids AS (
    SELECT DISTINCT pro_id FROM '{host_logs}' WHERE date <= DATE '2016-03-14'
),
holdout AS (
    SELECT
        CASE
            WHEN date = DATE '2016-03-15' THEN 'validation'
            ELSE 'test'
        END AS split_part,
        path,
        sys_call,
        pro_id,
        label
    FROM '{host_logs}'
    WHERE date >= DATE '2016-03-15'
)
SELECT
    split_part,
    count(*) FILTER (WHERE label = 1) AS anomalies,
    count(*) FILTER (WHERE label = 1 AND train_paths.path IS NULL) AS anomalies_unseen_path,
    count(*) FILTER (WHERE label = 1 AND train_sys_calls.sys_call IS NULL) AS anomalies_unseen_sys_call,
    count(*) FILTER (WHERE label = 1 AND train_path_sys_calls.path IS NULL) AS anomalies_unseen_path_sys_call,
    round(100.0 * count(*) FILTER (WHERE label = 1 AND train_path_sys_calls.path IS NULL) / count(*) FILTER (WHERE label = 1), 4) AS unseen_path_sys_call_pct,
    count(*) FILTER (WHERE label = 1 AND train_pro_ids.pro_id IS NULL) AS anomalies_unseen_pro_id,
    round(100.0 * count(*) FILTER (WHERE label = 1 AND train_pro_ids.pro_id IS NULL) / count(*) FILTER (WHERE label = 1), 4) AS unseen_pro_id_pct
FROM holdout
LEFT JOIN train_paths USING (path)
LEFT JOIN train_sys_calls USING (sys_call)
LEFT JOIN train_path_sys_calls USING (path, sys_call)
LEFT JOIN train_pro_ids USING (pro_id)
GROUP BY 1
ORDER BY CASE split_part WHEN 'validation' THEN 1 ELSE 2 END
""")


**Inference.** Holdout drift is low for `path` and `sys_call`, but non-trivial for `pro_id`. That validates the decision to aggregate `pro_id` rather than treat it as a categorical identity.


### Baseline Feature Shortlist

Keep as baseline raw inputs:

- `path`
- `sys_call`
- `time` only through derived features

Use for derived features only:

- `pro_id`
- `date`

Recommended first derived features:

- `hour`, `minute`, `seconds_since_midnight`, `hour_block`
- `path_train_count`, `path_train_relative_freq`
- `sys_call_train_count`, `sys_call_train_relative_freq`
- `path_sys_call_train_count`, `path_sys_call_train_relative_freq`
- `pro_id_train_count`, `pro_id_train_distinct_path_count`, `pro_id_train_distinct_sys_call_count`
- `is_rare_path`, `is_rare_sys_call`, `is_rare_path_sys_call`, `is_low_history_pro_id`

Exclude from baseline features:

- `attack_cat`
- `attack_subcat`
- `label`
- `event_id`
- raw `pro_id` as a categorical identity
- raw `date` as a default feature

Implementation rule:

- fit all count, frequency, and rarity lookups on the training split only
- join those learned statistics into validation and test
- map unseen holdout values to a neutral default such as `0` or `unknown`


## Phase 6: Visualization Pack

This section turns the earlier phase outputs into a compact visual pack for reporting and review. It includes schema and null-rate tables, class balance, time-based charts, top entity views, and a distinct-count summary.


In [ ]:
host_schema_df = query_df(f"DESCRIBE SELECT * FROM '{host_logs}'")
host_schema_df["dataset"] = "host_logs"
ground_truth_schema_df = query_df(f"DESCRIBE SELECT * FROM '{ground_truth}'")
ground_truth_schema_df["dataset"] = "ground_truth"
schema_summary_df = pd.concat([host_schema_df, ground_truth_schema_df], ignore_index=True)
schema_summary_df = schema_summary_df[["dataset", "column_name", "column_type", "null", "key", "default", "extra"]]
schema_summary_df


**Inference.** This table is the notebook-level reporting view of the validated schemas from Phase 1.


In [ ]:
null_rate_df = pd.concat(
    [
        null_rate_table(
            f"'{host_logs}'",
            "host_logs",
            ["date", "time", "pro_id", "path", "sys_call", "event_id", "attack_cat", "attack_subcat", "label"],
        ),
        null_rate_table(
            f"'{ground_truth}'",
            "ground_truth",
            ["date", "time", "attack_cat", "attack_subcat", "attack_name", "attack_refrence", "ips"],
        ),
    ],
    ignore_index=True,
)
null_rate_df = null_rate_df.sort_values(["dataset", "column_name"]).reset_index(drop=True)
null_rate_df


**Inference.** Nulls are not the main problem in either dataset. The real issues are structural corruption in `ground_truth` and leakage-prone labels in `host_logs`.


In [ ]:
class_balance_df = query_df(f"""
SELECT CAST(label AS VARCHAR) AS label, count(*) AS row_count
FROM '{host_logs}'
GROUP BY 1
ORDER BY 1
""")
ax = class_balance_df.plot(kind="bar", x="label", y="row_count", figsize=(7, 4), legend=False, color=["#6c8ebf", "#d97a42"])
ax.set_title("Host Log Class Balance")
ax.set_xlabel("label")
ax.set_ylabel("row count")
plt.tight_layout()
plt.show()
class_balance_df


**Inference.** Class imbalance is real, so accuracy is not a useful training metric. PR-oriented metrics will matter more than raw accuracy later on.


In [ ]:
daily_volume_df = query_df(f"""
SELECT
    date,
    count(*) AS event_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct
FROM '{host_logs}'
GROUP BY 1
ORDER BY 1
""")
ax = daily_volume_df.plot(x="date", y=["event_count", "anomaly_count"], figsize=(10, 4), marker="o")
ax.set_title("Event Volume And Anomaly Volume By Date")
ax.set_xlabel("date")
ax.set_ylabel("row count")
plt.tight_layout()
plt.show()
daily_volume_df


**Inference.** The daily volume view makes the campaign progression visible and reinforces the need for time-aware splits.


In [ ]:
hourly_rate_df = query_df(f"""
SELECT
    CAST(EXTRACT('hour' FROM time) AS INTEGER) AS hour,
    count(*) AS event_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct
FROM '{host_logs}'
GROUP BY 1
ORDER BY 1
""")
ax = hourly_rate_df.plot(kind="bar", x="hour", y="anomaly_rate_pct", figsize=(10, 4), legend=False, color="#4c956c")
ax.set_title("Hourly Anomaly Rate")
ax.set_xlabel("hour")
ax.set_ylabel("anomaly rate (%)")
plt.tight_layout()
plt.show()
hourly_rate_df


**Inference.** This chart highlights the strongest temporal shortcut in the dataset: anomalies cluster in a narrow hourly band.


In [ ]:
top_paths_df = query_df(f"""
SELECT
    path,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct,
    count(*) AS event_count
FROM '{host_logs}'
GROUP BY 1
ORDER BY anomaly_count DESC, event_count DESC
LIMIT 15
""")
ax = top_paths_df.sort_values("anomaly_count").plot(kind="barh", x="path", y="anomaly_count", figsize=(10, 6), legend=False, color="#bc4749")
ax.set_title("Top Paths By Anomaly Count")
ax.set_xlabel("anomaly count")
ax.set_ylabel("path")
plt.tight_layout()
plt.show()
top_paths_df


**Inference.** Most anomaly volume is concentrated in a few executables rather than spread across the process landscape.


In [ ]:
top_path_rate_df = query_df(f"""
SELECT
    path,
    count(*) AS event_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct
FROM '{host_logs}'
GROUP BY 1
HAVING count(*) >= 100000
ORDER BY anomaly_rate_pct DESC, event_count DESC
LIMIT 15
""")
ax = top_path_rate_df.sort_values("anomaly_rate_pct").plot(kind="barh", x="path", y="anomaly_rate_pct", figsize=(10, 6), legend=False, color="#f4a259")
ax.set_title("Anomaly Rate By High-Volume Path")
ax.set_xlabel("anomaly rate (%)")
ax.set_ylabel("path")
plt.tight_layout()
plt.show()
top_path_rate_df


**Inference.** High-volume path anomaly rates show that some executables are nearly deterministic anomaly carriers.


In [ ]:
top_pro_id_df = query_df(f"""
SELECT
    pro_id,
    count(*) AS event_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_count,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct
FROM '{host_logs}'
GROUP BY 1
HAVING count(*) >= 10000
ORDER BY anomaly_rate_pct DESC, event_count DESC
LIMIT 15
""")
top_pro_id_df["pro_id"] = top_pro_id_df["pro_id"].astype(str)
ax = top_pro_id_df.sort_values("anomaly_rate_pct").plot(kind="barh", x="pro_id", y="anomaly_rate_pct", figsize=(10, 6), legend=False, color="#8d6a9f")
ax.set_title("Anomaly Rate By High-Volume pro_id")
ax.set_xlabel("anomaly rate (%)")
ax.set_ylabel("pro_id")
plt.tight_layout()
plt.show()
top_pro_id_df


**Inference.** Even when `pro_id` looks predictive, it should still be interpreted as a local execution handle, not as a stable semantic identity.


In [ ]:
distinct_count_df = distinct_count_table(
    f"'{host_logs}'",
    [
        ("pro_id", "pro_id"),
        ("path", "path"),
        ("sys_call", "sys_call"),
        ("event_id", "event_id"),
        ("attack_cat", "attack_cat"),
        ("attack_subcat", "attack_subcat"),
    ],
)
ax = distinct_count_df.sort_values("distinct_count").plot(kind="barh", x="feature", y="distinct_count", figsize=(8, 4), legend=False, color="#577590")
ax.set_title("Distinct Count Summary For Key Host-Log Columns")
ax.set_xlabel("distinct count")
ax.set_ylabel("feature")
plt.tight_layout()
plt.show()
distinct_count_df


**Inference.** The distinct-count summary explains the baseline feature choices: `event_id` is identifier-like, `path` and `sys_call` are usable, and `pro_id` needs aggregation.


## Sequential Phase S1: Ordering Reliability

Goal: verify whether the event stream can be ordered consistently enough for sequence modeling.


In [ ]:
s1_collision_df = query_df(f"""
SELECT
    'date, time' AS key_level,
    count(*) AS distinct_keys,
    count(*) FILTER (WHERE rows_per_key > 1) AS collided_keys,
    round(100.0 * count(*) FILTER (WHERE rows_per_key > 1) / count(*), 4) AS collided_pct,
    max(rows_per_key) AS max_rows_same_key,
    approx_quantile(rows_per_key, [0.5, 0.9, 0.99]) AS rows_per_key_quantiles
FROM (
    SELECT date, time, count(*) AS rows_per_key
    FROM '{host_logs}'
    GROUP BY 1, 2
)
UNION ALL
SELECT
    'date, time, pro_id' AS key_level,
    count(*) AS distinct_keys,
    count(*) FILTER (WHERE rows_per_key > 1) AS collided_keys,
    round(100.0 * count(*) FILTER (WHERE rows_per_key > 1) / count(*), 4) AS collided_pct,
    max(rows_per_key) AS max_rows_same_key,
    approx_quantile(rows_per_key, [0.5, 0.9, 0.99]) AS rows_per_key_quantiles
FROM (
    SELECT date, time, pro_id, count(*) AS rows_per_key
    FROM '{host_logs}'
    GROUP BY 1, 2, 3
)
UNION ALL
SELECT
    'date, time, pro_id, path' AS key_level,
    count(*) AS distinct_keys,
    count(*) FILTER (WHERE rows_per_key > 1) AS collided_keys,
    round(100.0 * count(*) FILTER (WHERE rows_per_key > 1) / count(*), 4) AS collided_pct,
    max(rows_per_key) AS max_rows_same_key,
    approx_quantile(rows_per_key, [0.5, 0.9, 0.99]) AS rows_per_key_quantiles
FROM (
    SELECT date, time, pro_id, path, count(*) AS rows_per_key
    FROM '{host_logs}'
    GROUP BY 1, 2, 3, 4
)
""")
s1_collision_df


**Inference.** Second-level timestamps collide almost everywhere, even inside `pro_id` and `(pro_id, path)` streams. A usable sequence order needs an explicit tie-breaker beyond `date + time`.


In [ ]:
s1_event_order_df = query_df(f"""
WITH ordered AS (
    SELECT
        pro_id,
        date,
        time,
        event_id,
        lag(event_id) OVER (PARTITION BY pro_id ORDER BY date, time, event_id) AS prev_event_id
    FROM '{host_logs}'
),
event_id_counts AS (
    SELECT
        count(*) AS distinct_event_ids,
        count(*) FILTER (WHERE n > 1) AS repeated_event_ids,
        max(n) AS max_rows_per_event_id,
        sum(n - 1) FILTER (WHERE n > 1) AS extra_rows_from_repeated_event_ids
    FROM (
        SELECT event_id, count(*) AS n
        FROM '{host_logs}'
        GROUP BY 1
    )
),
repeated_local_keys AS (
    SELECT
        count(*) FILTER (WHERE n > 1) AS repeated_proid_timestamp_event_keys,
        sum(n - 1) FILTER (WHERE n > 1) AS extra_rows_from_repeated_local_keys
    FROM (
        SELECT date, time, pro_id, event_id, count(*) AS n
        FROM '{host_logs}'
        GROUP BY 1, 2, 3, 4
    )
)
SELECT
    count(*) FILTER (WHERE prev_event_id IS NOT NULL) AS comparable_rows,
    count(*) FILTER (WHERE prev_event_id IS NOT NULL AND event_id > prev_event_id) AS event_id_increases,
    round(
        100.0 * count(*) FILTER (WHERE prev_event_id IS NOT NULL AND event_id > prev_event_id)
        / count(*) FILTER (WHERE prev_event_id IS NOT NULL),
        4
    ) AS event_id_increase_pct,
    count(*) FILTER (WHERE prev_event_id IS NOT NULL AND event_id = prev_event_id) AS event_id_ties,
    round(
        100.0 * count(*) FILTER (WHERE prev_event_id IS NOT NULL AND event_id = prev_event_id)
        / count(*) FILTER (WHERE prev_event_id IS NOT NULL),
        4
    ) AS event_id_tie_pct,
    count(*) FILTER (WHERE prev_event_id IS NOT NULL AND event_id < prev_event_id) AS event_id_decreases,
    event_id_counts.repeated_event_ids,
    event_id_counts.max_rows_per_event_id,
    event_id_counts.extra_rows_from_repeated_event_ids,
    repeated_local_keys.repeated_proid_timestamp_event_keys,
    repeated_local_keys.extra_rows_from_repeated_local_keys
FROM ordered
CROSS JOIN event_id_counts
CROSS JOIN repeated_local_keys
""")
s1_event_order_df


**Inference.** `event_id` is a workable local tie-breaker. Its ties line up with duplicated rows, and true decreases are effectively absent. The sequence sort rule should be `(date, time, event_id)` after dropping exact duplicates.


In [ ]:
s1_event_id_decrease_examples_df = query_df(f"""
WITH ordered AS (
    SELECT
        pro_id,
        path,
        date,
        time,
        event_id,
        sys_call,
        label,
        lag(event_id) OVER (PARTITION BY pro_id ORDER BY date, time, event_id) AS prev_event_id,
        lag(path) OVER (PARTITION BY pro_id ORDER BY date, time, event_id) AS prev_path,
        lag(date) OVER (PARTITION BY pro_id ORDER BY date, time, event_id) AS prev_date,
        lag(time) OVER (PARTITION BY pro_id ORDER BY date, time, event_id) AS prev_time,
        lag(sys_call) OVER (PARTITION BY pro_id ORDER BY date, time, event_id) AS prev_sys_call,
        lag(label) OVER (PARTITION BY pro_id ORDER BY date, time, event_id) AS prev_label
    FROM '{host_logs}'
)
SELECT
    prev_date,
    prev_time,
    prev_event_id,
    prev_path,
    prev_sys_call,
    prev_label,
    date,
    time,
    event_id,
    path,
    sys_call,
    label,
    pro_id
FROM ordered
WHERE prev_event_id IS NOT NULL AND event_id < prev_event_id
ORDER BY pro_id, date, time, event_id
""")
s1_event_id_decrease_examples_df


**Inference.** The only two observed reversals are normal `/usr/sbin/cron` rows one second apart. That is too small to undermine sequence modeling.


## Sequential Phase S2: Sequence Boundary Definition

Goal: choose the sequence unit before any sequential model is built.


In [ ]:
s2_boundary_summary_df = query_df(f"""
WITH proid AS (
    SELECT
        pro_id,
        count(*) AS seq_len,
        count(DISTINCT path) AS distinct_paths,
        min(date) AS min_date,
        max(date) AS max_date
    FROM '{host_logs}'
    GROUP BY 1
),
proid_path AS (
    SELECT
        pro_id,
        path,
        count(*) AS seq_len,
        min(date) AS min_date,
        max(date) AS max_date
    FROM '{host_logs}'
    GROUP BY 1, 2
)
SELECT
    'pro_id' AS boundary,
    count(*) AS sequence_count,
    round(avg(seq_len), 4) AS avg_seq_len,
    max(seq_len) AS max_seq_len,
    approx_quantile(seq_len, [0.5, 0.9, 0.99]) AS seq_len_quantiles,
    count(*) FILTER (WHERE seq_len >= 32) AS seq_ge_32,
    count(*) FILTER (WHERE seq_len >= 64) AS seq_ge_64,
    count(*) FILTER (WHERE seq_len >= 128) AS seq_ge_128,
    round(100.0 * sum(seq_len) FILTER (WHERE seq_len >= 64) / sum(seq_len), 4) AS rows_covered_ge_64_pct,
    round(100.0 * count(*) FILTER (WHERE distinct_paths > 1) / count(*), 4) AS multi_path_sequence_pct,
    round(100.0 * sum(seq_len) FILTER (WHERE distinct_paths > 1) / sum(seq_len), 4) AS multi_path_row_pct,
    count(*) FILTER (WHERE min_date <= DATE '2016-03-14' AND max_date >= DATE '2016-03-15') AS cross_train_validation,
    count(*) FILTER (WHERE min_date <= DATE '2016-03-15' AND max_date >= DATE '2016-03-16') AS cross_validation_test
FROM proid
UNION ALL
SELECT
    '(pro_id, path)' AS boundary,
    count(*) AS sequence_count,
    round(avg(seq_len), 4) AS avg_seq_len,
    max(seq_len) AS max_seq_len,
    approx_quantile(seq_len, [0.5, 0.9, 0.99]) AS seq_len_quantiles,
    count(*) FILTER (WHERE seq_len >= 32) AS seq_ge_32,
    count(*) FILTER (WHERE seq_len >= 64) AS seq_ge_64,
    count(*) FILTER (WHERE seq_len >= 128) AS seq_ge_128,
    round(100.0 * sum(seq_len) FILTER (WHERE seq_len >= 64) / sum(seq_len), 4) AS rows_covered_ge_64_pct,
    NULL AS multi_path_sequence_pct,
    NULL AS multi_path_row_pct,
    count(*) FILTER (WHERE min_date <= DATE '2016-03-14' AND max_date >= DATE '2016-03-15') AS cross_train_validation,
    count(*) FILTER (WHERE min_date <= DATE '2016-03-15' AND max_date >= DATE '2016-03-16') AS cross_validation_test
FROM proid_path
""")
s2_boundary_summary_df


**Inference.** `(pro_id, path)` is the cleaner sequence identity. `pro_id` alone is longer, but it collapses multiple executable contexts together, which works against the goal of learning transferable behavior rather than application identity.


In [ ]:
s2_proid_path_switch_examples_df = query_df(f"""
WITH seq AS (
    SELECT
        pro_id,
        count(*) AS seq_len,
        count(DISTINCT path) AS distinct_paths,
        list(DISTINCT path ORDER BY path) AS sample_paths
    FROM '{host_logs}'
    GROUP BY 1
)
SELECT pro_id, seq_len, distinct_paths, sample_paths
FROM seq
WHERE distinct_paths > 1
ORDER BY distinct_paths DESC, seq_len DESC
LIMIT 20
""")
s2_proid_path_switch_examples_df


**Inference.** These mixed `pro_id` streams confirm that raw `pro_id` is not a stable process identity over the whole capture. The first sequential baseline should use `(pro_id, path)` windows with candidate lengths `32` and `64`, and truncate sequences at split boundaries.


## Sequential Phase S3: Label Locality And Run Structure

Goal: determine whether anomalies behave like isolated points or local runs inside `(pro_id, path)` sequences.


In [ ]:
s3_sequence_composition_df = query_df(f"""
WITH seq AS (
    SELECT
        pro_id,
        path,
        count(*) AS seq_len,
        sum(label) AS anomaly_rows
    FROM '{host_logs}'
    GROUP BY 1, 2
)
SELECT
    count(*) AS sequence_count,
    count(*) FILTER (WHERE anomaly_rows = 0) AS normal_only_sequences,
    count(*) FILTER (WHERE anomaly_rows = seq_len) AS anomaly_only_sequences,
    count(*) FILTER (WHERE anomaly_rows > 0 AND anomaly_rows < seq_len) AS mixed_sequences,
    sum(seq_len) FILTER (WHERE anomaly_rows = 0) AS rows_in_normal_only_sequences,
    sum(seq_len) FILTER (WHERE anomaly_rows = seq_len) AS rows_in_anomaly_only_sequences,
    sum(seq_len) FILTER (WHERE anomaly_rows > 0 AND anomaly_rows < seq_len) AS rows_in_mixed_sequences
FROM seq
""")
s3_sequence_composition_df


**Inference.** Most `(pro_id, path)` sequences are normal-only, but almost all rows live inside a small number of very large mixed sequences. That pushes the modeling problem toward local window labeling instead of one label for an entire stream.


In [ ]:
s3_run_summary_df = query_df(f"""
WITH ordered AS (
    SELECT
        pro_id,
        path,
        date,
        time,
        event_id,
        label,
        lag(label) OVER (PARTITION BY pro_id, path ORDER BY date, time, event_id) AS prev_label,
        lead(label) OVER (PARTITION BY pro_id, path ORDER BY date, time, event_id) AS next_label
    FROM '{host_logs}'
)
SELECT
    count(*) FILTER (WHERE label = 1 AND coalesce(prev_label, 0) = 0) AS anomaly_run_count,
    count(*) FILTER (WHERE label = 1 AND coalesce(prev_label, 0) = 0 AND coalesce(next_label, 0) = 0) AS single_event_runs,
    count(*) FILTER (WHERE label = 1 AND coalesce(prev_label, 0) = 0 AND coalesce(next_label, 0) = 1) AS multi_event_runs,
    round(
        1.0 * count(*) FILTER (WHERE label = 1)
        / count(*) FILTER (WHERE label = 1 AND coalesce(prev_label, 0) = 0),
        4
    ) AS avg_run_len
FROM ordered
""")
s3_run_summary_df


**Inference.** Anomalies usually arrive as runs, not isolated spikes. Single-event anomaly runs are rare, so sequence windows should have enough context to capture a burst once it starts.


In [ ]:
s3_anomaly_gap_df = query_df(f"""
WITH ordered AS (
    SELECT
        pro_id,
        path,
        row_number() OVER (PARTITION BY pro_id, path ORDER BY date, time, event_id) AS row_pos,
        label
    FROM '{host_logs}'
),
anomaly_rows AS (
    SELECT
        pro_id,
        path,
        row_pos,
        lag(row_pos) OVER (PARTITION BY pro_id, path ORDER BY row_pos) AS prev_anomaly_pos
    FROM ordered
    WHERE label = 1
)
SELECT
    count(*) FILTER (WHERE prev_anomaly_pos IS NOT NULL) AS comparable_anomaly_pairs,
    count(*) FILTER (WHERE prev_anomaly_pos IS NOT NULL AND row_pos - prev_anomaly_pos - 1 = 0) AS zero_gap_pairs,
    round(
        100.0 * count(*) FILTER (WHERE prev_anomaly_pos IS NOT NULL AND row_pos - prev_anomaly_pos - 1 = 0)
        / count(*) FILTER (WHERE prev_anomaly_pos IS NOT NULL),
        4
    ) AS zero_gap_pair_pct,
    approx_quantile(row_pos - prev_anomaly_pos - 1, [0.5, 0.9, 0.99]) AS gap_quantiles,
    max(row_pos - prev_anomaly_pos - 1) AS max_gap
FROM anomaly_rows
""")
s3_anomaly_gap_df


**Inference.** Consecutive anomalous rows are usually contiguous. That reinforces the case for window-level sequence modeling with positive windows defined by the presence of any anomalous event.


## Sequential Phase S4: Syscall Mapping And Transition Analysis

Goal: map `sys_call` ids to interpretable Linux syscall names and test whether transition patterns look transferable across applications.


In [ ]:
s4_mapping_sanity_df = query_df(f"""
WITH syscall_stats AS (
    SELECT
        sys_call,
        count(*) AS event_count,
        sum(label) AS anomaly_count
    FROM '{host_logs}'
    GROUP BY 1
)
SELECT
    s.sys_call,
    l.i386_name,
    l.x86_64_name,
    l.preferred_name,
    event_count,
    anomaly_count
FROM syscall_stats s
JOIN read_csv_auto('{syscall_lookup}') l USING (sys_call)
ORDER BY event_count DESC
LIMIT 20
""")
s4_mapping_sanity_df


**Inference.** The high-volume ids fit the Linux `i386` table much better than the `x86_64` table. The working lookup for sequence analysis should therefore prefer the `i386` names while keeping the `x86_64` names visible for audit.


In [ ]:
s4_top_syscall_rates_df = query_df(f"""
SELECT
    h.sys_call,
    l.preferred_name AS syscall_name,
    l.preferred_family AS syscall_family,
    count(*) AS event_count,
    sum(label) AS anomaly_count,
    round(100.0 * sum(label) / count(*), 4) AS anomaly_rate_pct,
    count(DISTINCT path) AS distinct_paths
FROM '{host_logs}' h
JOIN read_csv_auto('{syscall_lookup}') l USING (sys_call)
GROUP BY 1, 2, 3
HAVING count(*) >= 100000
ORDER BY anomaly_rate_pct DESC, anomaly_count DESC
LIMIT 20
""")
s4_top_syscall_rates_df


**Inference.** Under the `i386`-preferred mapping, the dominant signals are now semantically coherent: timing, socket, I/O, and event-loop syscalls account for most anomaly mass.


In [ ]:
s4_top_transition_rates_df = query_df(f"""
WITH ordered AS (
    SELECT
        pro_id,
        path,
        date,
        time,
        event_id,
        label,
        lag(sys_call) OVER (PARTITION BY pro_id, path ORDER BY date, time, event_id) AS prev_sys_call,
        sys_call
    FROM '{host_logs}'
)
SELECT
    p.preferred_name AS prev_syscall_name,
    c.preferred_name AS syscall_name,
    p.preferred_family AS prev_family,
    c.preferred_family AS syscall_family,
    count(*) AS transition_count,
    sum(label) AS anomaly_target_count,
    round(100.0 * sum(label) / count(*), 4) AS anomaly_target_rate_pct,
    count(DISTINCT path) FILTER (WHERE label = 1) AS anomaly_paths
FROM ordered o
JOIN read_csv_auto('{syscall_lookup}') p ON o.prev_sys_call = p.sys_call
JOIN read_csv_auto('{syscall_lookup}') c ON o.sys_call = c.sys_call
WHERE prev_sys_call IS NOT NULL
GROUP BY 1, 2, 3, 4
HAVING count(*) >= 10000
ORDER BY anomaly_target_rate_pct DESC, anomaly_target_count DESC
LIMIT 20
""")
s4_top_transition_rates_df


**Inference.** Order adds information beyond bag-of-events counts. High-rate anomalous transitions such as `socketcall <-> read` and `clock_gettime <-> epoll_wait` are sequence patterns, not just standalone syscall frequencies.


In [ ]:
s4_cross_path_transition_df = query_df(f"""
WITH ordered AS (
    SELECT
        pro_id,
        path,
        date,
        time,
        event_id,
        label,
        lag(sys_call) OVER (PARTITION BY pro_id, path ORDER BY date, time, event_id) AS prev_sys_call,
        sys_call
    FROM '{host_logs}'
)
SELECT
    p.preferred_name AS prev_syscall_name,
    c.preferred_name AS syscall_name,
    p.preferred_family AS prev_family,
    c.preferred_family AS syscall_family,
    count(*) AS anomalous_transition_rows,
    count(DISTINCT path) AS anomaly_paths
FROM ordered o
JOIN read_csv_auto('{syscall_lookup}') p ON o.prev_sys_call = p.sys_call
JOIN read_csv_auto('{syscall_lookup}') c ON o.sys_call = c.sys_call
WHERE prev_sys_call IS NOT NULL AND label = 1
GROUP BY 1, 2, 3, 4
HAVING count(*) >= 1000 AND count(DISTINCT path) >= 3
ORDER BY anomaly_paths DESC, anomalous_transition_rows DESC
LIMIT 20
""")
s4_cross_path_transition_df


**Inference.** Several anomalous transitions recur across many executables, which is the first concrete evidence that the sequence signal may transfer across applications instead of staying path-specific.


In [ ]:
s4_family_transition_df = query_df(f"""
WITH ordered AS (
    SELECT
        pro_id,
        path,
        date,
        time,
        event_id,
        label,
        lag(sys_call) OVER (PARTITION BY pro_id, path ORDER BY date, time, event_id) AS prev_sys_call,
        sys_call
    FROM '{host_logs}'
)
SELECT
    p.preferred_family AS prev_family,
    c.preferred_family AS syscall_family,
    count(*) AS transition_count,
    sum(label) AS anomaly_target_count,
    round(100.0 * sum(label) / count(*), 4) AS anomaly_target_rate_pct
FROM ordered o
JOIN read_csv_auto('{syscall_lookup}') p ON o.prev_sys_call = p.sys_call
JOIN read_csv_auto('{syscall_lookup}') c ON o.sys_call = c.sys_call
WHERE prev_sys_call IS NOT NULL
GROUP BY 1, 2
HAVING count(*) >= 10000
ORDER BY anomaly_target_rate_pct DESC, anomaly_target_count DESC
LIMIT 20
""")
s4_family_transition_df


**Inference.** Family-level transitions look even more stable than exact syscall names. That makes them a strong candidate for the first transferable sequence representation.


## Sequential Phase S5: Order Importance And Transfer Risk

Goal: test whether order still matters after bag-of-events composition is preserved, and whether anomaly motifs survive held-out application paths.


In [ ]:
s5_family_shuffle_df = query_df(f"""
WITH val_slice AS (
    SELECT *
    FROM '{host_logs}'
    WHERE date = DATE '2016-03-15'
),
ordered AS (
    SELECT
        pro_id,
        path,
        label,
        lag(sys_call) OVER (PARTITION BY pro_id, path ORDER BY date, time, event_id) AS prev_sys_call,
        sys_call
    FROM val_slice
),
ordered_family AS (
    SELECT
        p.preferred_family AS prev_family,
        c.preferred_family AS syscall_family,
        count(*) AS ordered_count,
        round(100.0 * sum(label) / count(*), 4) AS ordered_rate
    FROM ordered o
    JOIN read_csv_auto('{syscall_lookup}') p ON o.prev_sys_call = p.sys_call
    JOIN read_csv_auto('{syscall_lookup}') c ON o.sys_call = c.sys_call
    WHERE prev_sys_call IS NOT NULL
    GROUP BY 1, 2
),
shuffled AS (
    SELECT
        pro_id,
        path,
        label,
        lag(sys_call) OVER (PARTITION BY pro_id, path ORDER BY hash(event_id, time, sys_call)) AS prev_sys_call,
        sys_call
    FROM val_slice
),
shuffled_family AS (
    SELECT
        p.preferred_family AS prev_family,
        c.preferred_family AS syscall_family,
        count(*) AS shuffled_count,
        round(100.0 * sum(label) / count(*), 4) AS shuffled_rate
    FROM shuffled o
    JOIN read_csv_auto('{syscall_lookup}') p ON o.prev_sys_call = p.sys_call
    JOIN read_csv_auto('{syscall_lookup}') c ON o.sys_call = c.sys_call
    WHERE prev_sys_call IS NOT NULL
    GROUP BY 1, 2
)
SELECT
    o.prev_family,
    o.syscall_family,
    o.ordered_count,
    o.ordered_rate,
    s.shuffled_count,
    s.shuffled_rate,
    round(o.ordered_rate - s.shuffled_rate, 4) AS rate_lift_pct
FROM ordered_family o
JOIN shuffled_family s USING (prev_family, syscall_family)
WHERE o.ordered_count >= 5000
ORDER BY rate_lift_pct DESC, o.ordered_count DESC
LIMIT 25
""")
s5_family_shuffle_df


**Inference.** Some transition families lose a meaningful amount of anomaly rate once local order is destroyed, so part of the signal is genuinely sequential rather than purely bag-of-events.


In [ ]:
s5_exact_shuffle_df = query_df(f"""
WITH top_pairs AS (
    SELECT * FROM (VALUES
        ('ugetrlimit', 'fcntl64'),
        ('fcntl64', 'ugetrlimit'),
        ('socketcall', 'read'),
        ('read', 'socketcall'),
        ('clock_gettime', 'epoll_wait'),
        ('epoll_wait', 'clock_gettime'),
        ('poll', 'read'),
        ('read', 'write'),
        ('socketcall', 'socketcall'),
        ('clock_gettime', 'clock_gettime')
    ) AS t(prev_name, curr_name)
),
val_slice AS (
    SELECT *
    FROM '{host_logs}'
    WHERE date = DATE '2016-03-15'
),
ordered AS (
    SELECT
        label,
        lag(sys_call) OVER (PARTITION BY pro_id, path ORDER BY date, time, event_id) AS prev_sys_call,
        sys_call
    FROM val_slice
),
ordered_stats AS (
    SELECT
        p.preferred_name AS prev_name,
        c.preferred_name AS curr_name,
        count(*) AS ordered_count,
        round(100.0 * sum(label) / count(*), 4) AS ordered_rate
    FROM ordered o
    JOIN read_csv_auto('{syscall_lookup}') p ON o.prev_sys_call = p.sys_call
    JOIN read_csv_auto('{syscall_lookup}') c ON o.sys_call = c.sys_call
    JOIN top_pairs t ON p.preferred_name = t.prev_name AND c.preferred_name = t.curr_name
    WHERE prev_sys_call IS NOT NULL
    GROUP BY 1, 2
),
shuffled AS (
    SELECT
        label,
        lag(sys_call) OVER (PARTITION BY pro_id, path ORDER BY hash(event_id, time, sys_call)) AS prev_sys_call,
        sys_call
    FROM val_slice
),
shuffled_stats AS (
    SELECT
        p.preferred_name AS prev_name,
        c.preferred_name AS curr_name,
        count(*) AS shuffled_count,
        round(100.0 * sum(label) / count(*), 4) AS shuffled_rate
    FROM shuffled o
    JOIN read_csv_auto('{syscall_lookup}') p ON o.prev_sys_call = p.sys_call
    JOIN read_csv_auto('{syscall_lookup}') c ON o.sys_call = c.sys_call
    JOIN top_pairs t ON p.preferred_name = t.prev_name AND c.preferred_name = t.curr_name
    WHERE prev_sys_call IS NOT NULL
    GROUP BY 1, 2
)
SELECT
    o.prev_name,
    o.curr_name,
    o.ordered_count,
    o.ordered_rate,
    s.shuffled_count,
    s.shuffled_rate,
    round(o.ordered_rate - s.shuffled_rate, 4) AS rate_lift_pct
FROM ordered_stats o
JOIN shuffled_stats s USING (prev_name, curr_name)
ORDER BY rate_lift_pct DESC, o.ordered_count DESC
""")
s5_exact_shuffle_df


**Inference.** The strongest exact lifts, especially `socketcall <-> read`, shrink sharply under shuffle. That is direct evidence that some anomaly motifs depend on local order rather than just event membership.


In [ ]:
s5_holdout_overlap_df = query_df(f"""
WITH train_ordered AS (
    SELECT
        path,
        label,
        lag(sys_call) OVER (PARTITION BY pro_id, path ORDER BY date, time, event_id) AS prev_sys_call,
        sys_call
    FROM '{host_logs}'
    WHERE date <= DATE '2016-03-14'
),
train_anom AS (
    SELECT
        o.path,
        o.prev_sys_call,
        o.sys_call,
        p.preferred_name AS prev_name,
        c.preferred_name AS curr_name,
        p.preferred_family AS prev_family,
        c.preferred_family AS curr_family
    FROM train_ordered o
    JOIN read_csv_auto('{syscall_lookup}') p ON o.prev_sys_call = p.sys_call
    JOIN read_csv_auto('{syscall_lookup}') c ON o.sys_call = c.sys_call
    WHERE o.prev_sys_call IS NOT NULL AND o.label = 1
),
valtest_ordered AS (
    SELECT
        path,
        label,
        lag(sys_call) OVER (PARTITION BY pro_id, path ORDER BY date, time, event_id) AS prev_sys_call,
        sys_call
    FROM '{host_logs}'
    WHERE date >= DATE '2016-03-15'
),
valtest_anom AS (
    SELECT
        o.path,
        o.prev_sys_call,
        o.sys_call,
        p.preferred_name AS prev_name,
        c.preferred_name AS curr_name,
        p.preferred_family AS prev_family,
        c.preferred_family AS curr_family
    FROM valtest_ordered o
    JOIN read_csv_auto('{syscall_lookup}') p ON o.prev_sys_call = p.sys_call
    JOIN read_csv_auto('{syscall_lookup}') c ON o.sys_call = c.sys_call
    WHERE o.prev_sys_call IS NOT NULL AND o.label = 1
),
holdout_paths AS (
    SELECT * FROM (VALUES
        ('/usr/lib/libreoffice/program/soffice.bin'),
        ('/usr/lib/firefox/firefox'),
        ('/usr/sbin/apache2'),
        ('/usr/bin/compiz'),
        ('/usr/bin/Xorg')
    ) AS t(path)
),
train_other_exact AS (
    SELECT hp.path AS holdout_path, t.prev_sys_call, t.sys_call
    FROM holdout_paths hp
    JOIN train_anom t ON t.path != hp.path
    GROUP BY 1, 2, 3
),
train_other_family AS (
    SELECT hp.path AS holdout_path, t.prev_family, t.curr_family
    FROM holdout_paths hp
    JOIN train_anom t ON t.path != hp.path
    GROUP BY 1, 2, 3
)
SELECT
    hp.path AS holdout_path,
    count(*) AS anomalous_transition_rows,
    count(*) FILTER (WHERE e.prev_sys_call IS NOT NULL) AS rows_with_seen_exact_transition,
    round(100.0 * count(*) FILTER (WHERE e.prev_sys_call IS NOT NULL) / count(*), 4) AS seen_exact_pct,
    count(*) FILTER (WHERE f.prev_family IS NOT NULL) AS rows_with_seen_family_transition,
    round(100.0 * count(*) FILTER (WHERE f.prev_family IS NOT NULL) / count(*), 4) AS seen_family_pct,
    count(DISTINCT row(v.prev_name, v.curr_name)) AS distinct_exact_transitions,
    count(DISTINCT row(v.prev_family, v.curr_family)) AS distinct_family_transitions
FROM holdout_paths hp
JOIN valtest_anom v ON v.path = hp.path
LEFT JOIN train_other_exact e ON e.holdout_path = hp.path AND e.prev_sys_call = v.prev_sys_call AND e.sys_call = v.sys_call
LEFT JOIN train_other_family f ON f.holdout_path = hp.path AND f.prev_family = v.prev_family AND f.curr_family = v.curr_family
GROUP BY 1
ORDER BY anomalous_transition_rows DESC
""")
s5_holdout_overlap_df


**Inference.** The anomalous transition vocabulary on the major application paths is almost entirely present on other paths during training. That is strong evidence that this dataset contains cross-application behavioral reuse, not only path-specific anomaly signatures.


## Sequential Phase S6: Modeling-Ready Sequence Specification

Goal: freeze the first trainable sequence dataset design so the next step can be implementation rather than more exploratory analysis.


In [ ]:
s6_window_support_df = query_df(f"""
WITH split_rows AS (
    SELECT
        CASE
            WHEN date <= DATE '2016-03-14' THEN 'train'
            WHEN date = DATE '2016-03-15' THEN 'validation'
            ELSE 'test'
        END AS split_name,
        pro_id,
        path,
        count(*) AS seq_len,
        sum(label) AS anomaly_rows
    FROM '{host_logs}'
    GROUP BY 1, 2, 3
)
SELECT
    split_name,
    count(*) AS sequence_fragments,
    sum(seq_len) AS total_rows,
    round(avg(seq_len), 4) AS avg_seq_len,
    approx_quantile(seq_len, [0.5, 0.9, 0.99]) AS seq_len_quantiles,
    count(*) FILTER (WHERE seq_len >= 32) AS fragments_ge_32,
    count(*) FILTER (WHERE seq_len >= 64) AS fragments_ge_64,
    sum(CASE WHEN seq_len >= 32 THEN floor((seq_len - 32) / 16) + 1 ELSE 0 END) AS windows_len32_stride16,
    sum(CASE WHEN seq_len >= 64 THEN floor((seq_len - 64) / 32) + 1 ELSE 0 END) AS windows_len64_stride32,
    sum(CASE WHEN seq_len >= 64 THEN floor((seq_len - 64) / 64) + 1 ELSE 0 END) AS windows_len64_stride64
FROM split_rows
GROUP BY 1
ORDER BY CASE split_name WHEN 'train' THEN 1 WHEN 'validation' THEN 2 ELSE 3 END
""")
s6_window_support_df


**Inference.** The default `64/32` design is well supported in every split, with about `2.12M` train windows and roughly `390k` / `299k` validation/test windows. That is enough coverage for a first sequence baseline without shrinking the window further by default.


In [ ]:
s6_gap_support_df = query_df(f"""
WITH ordered AS (
    SELECT
        CASE
            WHEN date <= DATE '2016-03-14' THEN 'train'
            WHEN date = DATE '2016-03-15' THEN 'validation'
            ELSE 'test'
        END AS split_name,
        pro_id,
        path,
        date + time AS event_ts,
        lag(date + time) OVER (
            PARTITION BY CASE
                WHEN date <= DATE '2016-03-14' THEN 'train'
                WHEN date = DATE '2016-03-15' THEN 'validation'
                ELSE 'test'
            END, pro_id, path
            ORDER BY date, time, event_id
        ) AS prev_ts
    FROM '{host_logs}'
),
gaps AS (
    SELECT
        split_name,
        datediff('second', prev_ts, event_ts) AS gap_seconds
    FROM ordered
    WHERE prev_ts IS NOT NULL
)
SELECT
    split_name,
    count(*) AS comparable_rows,
    count(*) FILTER (WHERE gap_seconds = 0) AS zero_second_gaps,
    round(100.0 * count(*) FILTER (WHERE gap_seconds = 0) / count(*), 4) AS zero_second_gap_pct,
    approx_quantile(gap_seconds, [0.5, 0.9, 0.99]) AS gap_quantiles,
    max(gap_seconds) AS max_gap_seconds
FROM gaps
GROUP BY 1
ORDER BY CASE split_name WHEN 'train' THEN 1 WHEN 'validation' THEN 2 ELSE 3 END
""")
s6_gap_support_df


**Inference.** Inter-event timing is extremely skewed: more than `96%` of consecutive events occur in the same second, while the long tail is huge. The sequence dataset should therefore keep `delta_seconds`, but feed the model a clipped `log1p` transform rather than the raw scale.


**Final Spec.** The first sequence dataset should use `(pro_id, path)` fragments, sort by `(date, time, event_id)`, split by date before windowing, and build fixed windows labeled positive if any event is anomalous.

Default design:
- window length `64`
- stride `32`
- exact syscall id from the `i386`-preferred lookup
- syscall family from the `8` coarse families
- `delta_seconds_log1p_clipped`
- exclude raw `path`, raw `pro_id`, `event_id`, absolute time, and attack metadata from model inputs

The full implementation-ready spec is recorded in [sequence-dataset-spec.md](/Users/harrish/Desktop/practicum/anomaly-detection-xai/docs/sequence-dataset-spec.md:1).


## Phase 7: Feature Engineering And LSTM Baseline

This section converts the sequence EDA decisions into a trainable LSTM dataset and model. The pipeline keeps `path` and `pro_id` as sequence-boundary fields only; they are not model inputs.


### Analysis Question

Can we build leakage-safe fixed-length sequence windows for a first LSTM anomaly detector?


In [ ]:
import json

project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sequence_output = dataset_root / "lstm_sequences"
lstm_model_output = dataset_root / "lstm_model"
build_sequence_script = project_root / "scripts" / "build_lstm_sequence_dataset.py"
train_lstm_script = project_root / "scripts" / "train_lstm.py"

print(f"project_root: {project_root}")
print(f"sequence_output: {sequence_output}")
print(f"lstm_model_output: {lstm_model_output}")
print(f"build script: {build_sequence_script}")
print(f"train script: {train_lstm_script}")


**Short inference.** The feature engineering output is stored under the dataset folder because it is generated data. The model output is also local/generated state and should not be committed.


In [ ]:
# Build a sampled sequence dataset for the first LSTM baseline.
# Increase the max window counts later if training is stable and runtime is acceptable.
subprocess.run(
    [
        sys.executable,
        str(build_sequence_script),
        "--dataset-root",
        str(dataset_root),
        "--output-dir",
        str(sequence_output),
        "--window-length",
        "64",
        "--stride",
        "32",
        "--negative-ratio",
        "2",
        "--max-positive-windows",
        "20000",
        "--max-validation-positive-windows",
        "5000",
        "--max-test-positive-windows",
        "5000",
        "--overwrite",
    ],
    check=True,
)


**Short inference.** The generated windows use syscall token, syscall-family token, and clipped log delta-time. Leakage fields such as `attack_cat`, `attack_subcat`, raw `path`, raw `pro_id`, and `event_id` are excluded from model inputs.


In [ ]:
sequence_metadata = json.loads((sequence_output / "metadata.json").read_text())
pd.DataFrame(sequence_metadata["splits"])


**Short inference.** This table confirms how many positive and negative sequence windows were sampled for each split before LSTM training.


### Analysis Question

Can a first LSTM learn anomaly patterns from the engineered syscall sequence windows?


In [ ]:
tensorflow = ensure_package("tensorflow")

subprocess.run(
    [
        sys.executable,
        str(train_lstm_script),
        "--sequence-dir",
        str(sequence_output),
        "--output-dir",
        str(lstm_model_output),
        "--epochs",
        "5",
        "--batch-size",
        "256",
        "--overwrite",
    ],
    check=True,
)


**Short inference.** The first LSTM baseline is intentionally small. Its role is to verify that the engineered sequence representation is trainable before moving to explainability experiments.


In [ ]:
lstm_metrics = json.loads((lstm_model_output / "metrics.json").read_text())
summary_rows = []
for split_name in ["validation", "test"]:
    threshold_metrics = lstm_metrics[f"{split_name}_threshold_metrics"]
    summary_rows.append({
        "split": split_name,
        "threshold": threshold_metrics["threshold"],
        "precision": threshold_metrics["precision"],
        "recall": threshold_metrics["recall"],
        "f1": threshold_metrics["f1"],
        "tp": threshold_metrics["tp"],
        "fp": threshold_metrics["fp"],
        "fn": threshold_metrics["fn"],
        "tn": threshold_metrics["tn"],
    })
pd.DataFrame(summary_rows)


**Short inference.** Use validation performance to tune model and threshold choices. Treat test performance as the final held-out estimate after the setup is frozen.
